# KisaanMitra — Crop Disease Detection
**Model:** EfficientNetB3 | **Classes:** 17 | **Images:** 25,828

---

## Section 1 — Setup

In [ ]:
# Cell 1 — Imports + GPU Check
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.models import load_model

print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU available:', gpus if gpus else '❌ No GPU — enable T4 in Runtime settings!')

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Cell 3 — Copy Training Data to Local Disk (~16 mins)
import shutil

DRIVE_PATH = '/content/drive/MyDrive/Athena/Kissan'
LOCAL_DATA = '/content/TrainingData'

if os.path.exists(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)  # clean any partial copy

shutil.copytree(f'{DRIVE_PATH}/TrainingData', LOCAL_DATA)
print('Data copied ✅')

In [ ]:
# Cell 4 — Verify Copy
total = sum(len(files) for _, _, files in os.walk(LOCAL_DATA))
print(f'Files found: {total}')  # Should be 25,828
assert total == 25828, f'⚠️ Expected 25828 files, got {total}'

---
## Section 2 — Data Pipeline

In [ ]:
# Cell 5 — Load CSVs, Config & All Pipeline Functions
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)

# Paths
DRIVE_PATH = '/content/drive/MyDrive/Athena/Kissan'
LOCAL_DATA = '/content/TrainingData'
OLD_CSV_PATH = '/content/drive/MyDrive/Athena/Kissan/TrainingData'

# Load CSVs
train_df = pd.read_csv(f'{DRIVE_PATH}/train.csv')
val_df   = pd.read_csv(f'{DRIVE_PATH}/val.csv')
test_df  = pd.read_csv(f'{DRIVE_PATH}/test.csv')

# Fix filepaths to point to local disk
for df in [train_df, val_df, test_df]:
    df['filepath'] = df['filepath'].str.replace(OLD_CSV_PATH, LOCAL_DATA, regex=False)

print('Sample filepath:', train_df['filepath'].iloc[0])
print('File exists    :', os.path.exists(train_df['filepath'].iloc[0]))
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# Config
AUTOTUNE     = tf.data.AUTOTUNE
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
CLASS_NAMES  = sorted(train_df['label'].unique())
class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
idx_to_class = {v: k for k, v in class_to_idx.items()}
NUM_CLASSES  = len(CLASS_NAMES)

print(f'\nClasses ({NUM_CLASSES}):', CLASS_NAMES)

# --- Augmentation functions ---

def augment_standard(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_crop(
        tf.image.resize_with_crop_or_pad(img, 250, 250),
        size=[224, 224, 3]
    )
    return img, label

def augment_heavy(img, label):
    """Heavy augmentation for weak classes (Rice_Blast, Sugarcane_Healthy)."""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.25)
    img = tf.image.random_contrast(img, lower=0.7, upper=1.3)
    img = tf.image.random_saturation(img, lower=0.7, upper=1.3)
    img = tf.image.random_crop(
        tf.image.resize_with_crop_or_pad(img, 260, 260),
        size=[224, 224, 3]
    )
    return img, label

# Weak class config
WEAK_CLASSES = {'Rice_Blast', 'Sugarcane_Healthy'}
weak_indices = {class_to_idx[c] for c in WEAK_CLASSES}

# --- Dataset builder (targeted augmentation) ---

def df_to_dataset(df, training=False):
    """Builds tf.data pipeline with targeted heavy augmentation on weak classes."""
    filepaths    = df['filepath'].values
    label_indices = [class_to_idx[l] for l in df['label'].values]
    labels       = tf.keras.utils.to_categorical(label_indices, num_classes=NUM_CLASSES)
    is_weak      = [1 if idx in weak_indices else 0 for idx in label_indices]

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels, is_weak))
    if training:
        ds = ds.shuffle(buffer_size=2000)

    def load_and_route(filepath, label, is_weak):
        img = tf.io.read_file(filepath)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32)
        if training:
            img, label = tf.cond(
                tf.equal(is_weak, 1),
                lambda: augment_heavy(img, label),
                lambda: augment_standard(img, label)
            )
        return img, label

    ds = ds.map(load_and_route, num_parallel_calls=4)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

print('\nPipeline functions ready ✅')
print(f'Weak classes: {WEAK_CLASSES} → indices {weak_indices}')

---
## Section 3 — Phase 4 Training (Class-Weighted Loss)

In [ ]:
# Cell 6 — Build Datasets
train_ds = df_to_dataset(train_df, training=True)
val_ds   = df_to_dataset(val_df,   training=False)
test_ds  = df_to_dataset(test_df,  training=False)

print(f'Train batches : {len(train_ds)}')
print(f'Val batches   : {len(val_ds)}')
print(f'Test batches  : {len(test_ds)}')
print('Datasets ready ✅')

In [ ]:
# Cell 7 — Load Phase 1 Model + Unfreeze Top 30 Layers + Compile
model = load_model(
    f'{DRIVE_PATH}/kisaanmitra_phase1.keras',
    custom_objects={'preprocess_input': preprocess_input}
)

base_model = model.layers[2]  # EfficientNetB3
for layer in base_model.layers:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True

# Class weights — penalise mistakes on weak classes 3x more
class_weights = {i: 1.0 for i in range(NUM_CLASSES)}
class_weights[class_to_idx['Rice_Blast']]       = 3.0
class_weights[class_to_idx['Sugarcane_Healthy']] = 3.0

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainable = sum([w.numpy().size for w in model.trainable_weights])
print(f'Trainable params : {trainable:,}')
print(f'Class weights    : { {k:v for k,v in class_weights.items() if v > 1.0} }')
print('Model ready ✅')

In [ ]:
# Cell 8 — Train Phase 4
print('Starting Phase 4 training 🚀')

callbacks_p4 = [
    ModelCheckpoint(
        f'{DRIVE_PATH}/kisaanmitra_phase4.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.3,
        patience=3, min_lr=1e-7, verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    )
]

history_phase4 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_p4
)

print('Phase 4 complete ✅')
print(f'Best val_accuracy: {max(history_phase4.history["val_accuracy"]):.4f}')

---
## Section 4 — Evaluation

In [ ]:
# Cell 9 — Classification Report
from sklearn.metrics import classification_report

y_pred         = model.predict(test_ds, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true         = np.concatenate([np.argmax(y, axis=1) for x, y in test_ds])
class_names_list = [idx_to_class[i] for i in range(NUM_CLASSES)]

print('\n' + '='*60)
print('CLASSIFICATION REPORT — Phase 4')
print('='*60)
print(classification_report(y_true, y_pred_classes, target_names=class_names_list))

In [ ]:
# Cell 10 — Per-Class Accuracy Bar
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred_classes)
per_class_acc = cm.diagonal() / cm.sum(axis=1)

print('\nPer-Class Accuracy:')
print('='*50)
for name, acc in sorted(zip(class_names_list, per_class_acc), key=lambda x: x[1]):
    bar    = '█' * int(acc * 20)
    status = '⚠️' if acc < 0.95 else '✅'
    print(f'{status} {name:<30} {acc*100:6.2f}%  {bar}')

In [ ]:
# Cell 11 — Confusion Matrix Heatmap
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(16, 14))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names_list,
    yticklabels=class_names_list
)
plt.title('KisaanMitra — Confusion Matrix (Phase 4 Test Set)', fontsize=14)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/confusion_matrix_phase4.png', dpi=150)
plt.show()
print('Saved to Drive ✅')

In [ ]:
# Cell 12 — Training Curves
import matplotlib.pyplot as plt

acc      = history_phase4.history['accuracy']
val_acc  = history_phase4.history['val_accuracy']
loss     = history_phase4.history['loss']
val_loss = history_phase4.history['val_loss']
epochs   = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, acc,     'b-o', label='Train Accuracy')
ax1.plot(epochs, val_acc, 'r-o', label='Val Accuracy')
ax1.set_title('KisaanMitra Phase 4 — Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, loss,     'b-o', label='Train Loss')
ax2.plot(epochs, val_loss, 'r-o', label='Val Loss')
ax2.set_title('KisaanMitra Phase 4 — Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/phase4_curves.png', dpi=150)
plt.show()
print('Curves saved to Drive ✅')